# Inter-Annotator Agreement
Computes raw agreement, Cohen's κ, and Krippendorff's α between the two annotators.

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
import krippendorff
import matplotlib.pyplot as plt

In [ ]:
LEEN_PATH = "../../data/annotator_agreedment/manual_labels_v1_leen.csv"
V5_PATH   = "../../data/annotator_agreedment/manual_labels_v5.csv"

leen = pd.read_csv(LEEN_PATH)[["original_index", "manual_label"]].rename(columns={"manual_label": "label_leen"})
v5   = pd.read_csv(V5_PATH)[["original_index", "manual_label"]].rename(columns={"manual_label": "label_v5"})

df = leen.merge(v5, on="original_index")

# Exclude items where either annotator skipped
df = df[(df["label_leen"] != "skip") & (df["label_v5"] != "skip")].copy()

print(f"Items after merging (excl. skips): {len(df)}")
print()
print("Leen label distribution:")
print(df["label_leen"].value_counts())
print()
print("v5 label distribution:")
print(df["label_v5"].value_counts())

In [ ]:
# ── 1. Raw agreement ────────────────────────────────────────────────────────
agreed = (df["label_leen"] == df["label_v5"]).sum()
N = len(df)
p_raw = agreed / N

# ── 2. Cohen's kappa ────────────────────────────────────────────────────────
kappa = cohen_kappa_score(df["label_leen"], df["label_v5"])

# ── 3. Krippendorff's alpha ─────────────────────────────────────────────────
categories = sorted(set(df["label_leen"]) | set(df["label_v5"]))
cat2int = {c: i for i, c in enumerate(categories)}

reliability_data = [
    df["label_leen"].map(cat2int).tolist(),
    df["label_v5"].map(cat2int).tolist(),
]
alpha = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement="nominal")

print(f"N items:              {N}")
print(f"Items agreed:         {agreed}")
print()
print(f"Raw agreement (p):    {p_raw:.4f}")
print(f"Cohen's kappa (κ):    {kappa:.4f}")
print(f"Krippendorff's α:     {alpha:.4f}")

In [ ]:
cm = confusion_matrix(df["label_leen"], df["label_v5"], labels=categories)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=categories)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix\n(rows = Leen, cols = v5)")
ax.set_xlabel("Annotator v5")
ax.set_ylabel("Annotator Leen")
plt.tight_layout()
plt.show()

In [ ]:
disagreements = df[df["label_leen"] != df["label_v5"]].copy()
print(f"Total disagreements: {len(disagreements)}")
disagreements[["original_index", "label_leen", "label_v5"]]